#### 8. OutputFixingParser（输出修复解析器）

模型返回的 JSON 缺了个括号怎么办？这个解析器会尝试**自动修补**常见的格式错误（少引号、少逗号、多出注释等），尽量抢救回来，避免直接报错。

In [ ]:
import sys

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
# from langchain_core.pydantic_v1 import BaseModel, Field
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.exceptions import OutputParserException

# 先确认：当前 notebook 用的到底是不是你说的 `lang` 环境
# try:
#     import langchain
#     print("python:", sys.executable)
#     print("langchain version:", getattr(langchain, "__version__", "unknown"))
# except Exception as e:
#     print("python:", sys.executable)
#     raise

# ================= 1. 定义你期望的数据结构 =================
class Actor(BaseModel):
    name: str = Field(description="演员姓名")
    age: int = Field(description="演员年龄")
    movie: str = Field(description="代表作")

# ================= 2. 创建主解析器 =================
# 这个解析器要求严格的 JSON 格式
primary_parser = PydanticOutputParser(pydantic_object=Actor)

# ================= 3. 模拟一个“格式错误”的输出 =================
# 假设大模型没按规矩输出，输出了一个格式残缺的伪 JSON（缺了引号和括号）
bad_llm_output = """
这是你要的信息：
{ name: '周星驰', age: 60, movie: '少林足球' }
"""

# 验证一下：主解析器绝对会报错
try:
    primary_parser.parse(bad_llm_output)
except Exception as e:
    print(f"❌ 主解析器报错：\n{type(e).__name__}: {e}\n")

# ================= 4. 创建修复用的 LLM =================
# 注意：修复 LLM 通常建议用比较强的模型（如 GPT-4 或 GPT-3.5），因为改错需要一定的理解力
fixing_llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.0)

# ================= 5. 构建 OutputFixingParser =================
repair_parser = OutputFixingParser.from_llm(
    parser=primary_parser,      # 传入刚才报错的主解析器
    llm=fixing_llm              # 传入负责修错的 LLM
)

# ================= 6. 执行修复解析 =================
print("🚀 正在调用 OutputFixingParser 进行自动修复...\n")
fixed_result = repair_parser.parse(bad_llm_output)

print("✅ 修复成功！结果如下：")
print(fixed_result)
print(f"\n提取出的姓名: {fixed_result.name}")
print(f"数据类型: {type(fixed_result)}") # 注意：返回的依然是 Pydantic 对象

python: f:\A_SCU\Code\LangChainProject\lang\Scripts\python.exe
langchain version: 1.2.9
